# Inference Engine — Advanced

> **Section 01 — Topic 07 — advanced**

**Prerequisites:** `07-inference-engine/foundations.ipynb`

**What you'll learn:**
- Speculative decoding for faster generation
- Continuous batching and paged attention
- Advanced decoding variants

**What you'll build:**
- A speculative decoding implementation from scratch

## Setup

We'll use two GPT-2 models: the small (124M) as our "draft" model and the medium (355M) as our "target" model. In production, you'd use something like a 1B draft with a 70B target, but the principles are identical.

In [ ]:
!pip install -q torch transformers matplotlib numpy

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Draft model: GPT-2 small (124M)
draft_model_name = "gpt2"
draft_tokenizer = AutoTokenizer.from_pretrained(draft_model_name)
draft_model = AutoModelForCausalLM.from_pretrained(draft_model_name).to(device)
draft_model.eval()

# Target model: GPT-2 medium (355M)
target_model_name = "gpt2-medium"
target_tokenizer = AutoTokenizer.from_pretrained(target_model_name)
target_model = AutoModelForCausalLM.from_pretrained(target_model_name).to(device)
target_model.eval()

# They share the same tokenizer
tokenizer = draft_tokenizer
tokenizer.pad_token = tokenizer.eos_token

print(f"Draft model:  {draft_model_name} ({sum(p.numel() for p in draft_model.parameters()):,} params)")
print(f"Target model: {target_model_name} ({sum(p.numel() for p in target_model.parameters()):,} params)")

---
## 1. Speculative Decoding — Draft Model + Verification

Speculative decoding (Leviathan et al., 2023; Chen et al., 2023) is one of the most elegant inference optimization techniques. The core insight is that **verification is cheaper than generation**.

In standard autoregressive decoding, a large model generates one token at a time — each requiring a full forward pass. Speculative decoding uses a small, fast "draft" model to generate several candidate tokens, then runs a single forward pass of the large "target" model to verify all candidates simultaneously.

Here's the algorithm:
1. Use the draft model to generate `K` tokens autoregressively (fast, since the model is small)
2. Run the target model on the full sequence (prompt + K draft tokens) in a single forward pass
3. Compare the target model's probabilities with the draft model's choices:
   - If the target model assigns higher probability to the draft token than the draft model did, **accept** the token
   - If the target model assigns lower probability, accept with probability `p_target / p_draft` (rejection sampling)
   - Once a token is rejected, resample from an adjusted distribution and discard all subsequent draft tokens
4. Repeat from step 1

The key mathematical property is that this produces **exactly the same distribution** as sampling directly from the target model. It's not an approximation — it's lossless acceleration. The speedup comes from the fact that when the draft model agrees with the target model (which happens frequently for easy tokens like "the", "of", etc.), we get multiple tokens for the cost of one target model forward pass.

The speedup depends on the **acceptance rate** — how often the target model agrees with the draft model. With a well-chosen draft model, acceptance rates of 70-90% are common, yielding 2-3x speedup.

In [ ]:
def speculative_decode(
    target_model, draft_model, tokenizer, prompt,
    max_new_tokens=100, K=5, temperature=1.0
):
    """
    Speculative decoding: use draft model to propose K tokens,
    verify with target model in a single forward pass.
    
    Args:
        K: number of tokens to draft at each iteration
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    
    total_draft_tokens = 0
    total_accepted = 0
    n_generated = 0
    
    while n_generated < max_new_tokens:
        # === Step 1: Draft K tokens with the small model ===
        draft_tokens = []
        draft_probs = []
        draft_input = generated.clone()
        
        for _ in range(K):
            with torch.no_grad():
                draft_logits = draft_model(draft_input).logits[:, -1, :]
            
            if temperature > 0:
                draft_p = F.softmax(draft_logits / temperature, dim=-1)
                next_token = torch.multinomial(draft_p, num_samples=1)
            else:
                draft_p = F.softmax(draft_logits, dim=-1)
                next_token = torch.argmax(draft_logits, dim=-1, keepdim=True)
            
            token_prob = draft_p[0, next_token[0, 0]].item()
            draft_tokens.append(next_token[0, 0].item())
            draft_probs.append(token_prob)
            draft_input = torch.cat([draft_input, next_token], dim=-1)
        
        total_draft_tokens += K
        
        # === Step 2: Verify all K tokens with target model in ONE forward pass ===
        verify_input = torch.cat([
            generated,
            torch.tensor([draft_tokens], device=device)
        ], dim=-1)
        
        with torch.no_grad():
            target_logits = target_model(verify_input).logits
        
        # === Step 3: Accept/reject each draft token ===
        n_accepted = 0
        for i in range(K):
            # Position in the target output corresponding to this draft token
            pos = generated.shape[1] + i - 1
            target_p = F.softmax(target_logits[0, pos, :] / max(temperature, 1e-10), dim=-1)
            
            draft_token = draft_tokens[i]
            p_target = target_p[draft_token].item()
            p_draft = draft_probs[i]
            
            # Acceptance criterion: accept with probability min(1, p_target / p_draft)
            if p_draft > 0:
                accept_prob = min(1.0, p_target / p_draft)
            else:
                accept_prob = 1.0 if p_target > 0 else 0.0
            
            if torch.rand(1).item() < accept_prob:
                n_accepted += 1
                total_accepted += 1
            else:
                # Rejected — sample from adjusted distribution
                # p_adjusted = max(0, p_target - p_draft) normalized
                adjusted = torch.clamp(target_p - F.softmax(draft_logits / max(temperature, 1e-10), dim=-1)[0], min=0)
                if adjusted.sum() > 0:
                    adjusted = adjusted / adjusted.sum()
                    replacement = torch.multinomial(adjusted, num_samples=1)
                else:
                    replacement = torch.argmax(target_p, dim=-1, keepdim=True)
                
                # Accept tokens up to rejection point + the replacement
                accepted = draft_tokens[:n_accepted] + [replacement.item()]
                generated = torch.cat([
                    generated,
                    torch.tensor([accepted], device=device)
                ], dim=-1)
                n_generated += len(accepted)
                break
        else:
            # All K tokens accepted — also sample one bonus token from target
            pos = generated.shape[1] + K - 1
            bonus_p = F.softmax(target_logits[0, pos, :] / max(temperature, 1e-10), dim=-1)
            bonus_token = torch.multinomial(bonus_p, num_samples=1)
            
            all_tokens = draft_tokens + [bonus_token.item()]
            generated = torch.cat([
                generated,
                torch.tensor([all_tokens], device=device)
            ], dim=-1)
            n_generated += len(all_tokens)
        
        # Check for EOS
        if generated[0, -1].item() == tokenizer.eos_token_id:
            break
    
    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    acceptance_rate = total_accepted / total_draft_tokens if total_draft_tokens > 0 else 0
    
    return text, {
        "total_draft_tokens": total_draft_tokens,
        "total_accepted": total_accepted,
        "acceptance_rate": acceptance_rate,
        "tokens_generated": n_generated,
    }


# Test speculative decoding
prompt = "The theory of relativity describes"
text, stats = speculative_decode(
    target_model, draft_model, tokenizer, prompt,
    max_new_tokens=60, K=5, temperature=0.8
)

print("Generated text:")
print(text)
print(f"\nAcceptance rate: {stats['acceptance_rate']:.1%}")
print(f"Draft tokens: {stats['total_draft_tokens']}, Accepted: {stats['total_accepted']}")

In [ ]:
# Benchmark: speculative vs standard autoregressive
def standard_generate(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8):
    """Standard autoregressive generation with the target model."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated).logits[:, -1, :]
        if temperature > 0:
            probs = F.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        else:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=-1)
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(generated[0], skip_special_tokens=True)


# Warm up
_ = standard_generate(target_model, tokenizer, prompt, 10)
_ = speculative_decode(target_model, draft_model, tokenizer, prompt, 10, K=5)

# Benchmark
n_tokens = 80
n_runs = 3

# Standard (target model only)
std_times = []
for _ in range(n_runs):
    start = time.perf_counter()
    standard_generate(target_model, tokenizer, prompt, n_tokens, temperature=0.0)
    std_times.append(time.perf_counter() - start)

# Speculative
spec_times = []
spec_acceptance = []
for _ in range(n_runs):
    start = time.perf_counter()
    _, stats = speculative_decode(target_model, draft_model, tokenizer, prompt, n_tokens, K=5, temperature=0.0)
    spec_times.append(time.perf_counter() - start)
    spec_acceptance.append(stats['acceptance_rate'])

avg_std = np.mean(std_times)
avg_spec = np.mean(spec_times)

print(f"Standard decoding:    {avg_std:.3f}s")
print(f"Speculative decoding: {avg_spec:.3f}s")
print(f"Speedup:              {avg_std/avg_spec:.2f}x")
print(f"Avg acceptance rate:  {np.mean(spec_acceptance):.1%}")

In [ ]:
# Analyze acceptance rate across different K values
k_values = [1, 2, 3, 5, 8, 12]
k_acceptance_rates = []
k_speedups = []

for k in k_values:
    rates = []
    times = []
    for _ in range(3):
        start = time.perf_counter()
        _, stats = speculative_decode(
            target_model, draft_model, tokenizer, prompt,
            max_new_tokens=60, K=k, temperature=0.0
        )
        times.append(time.perf_counter() - start)
        rates.append(stats['acceptance_rate'])
    k_acceptance_rates.append(np.mean(rates))
    k_speedups.append(avg_std / np.mean(times))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(len(k_values)), k_acceptance_rates, tick_label=[str(k) for k in k_values], color='steelblue')
ax1.set_xlabel('K (draft tokens per iteration)')
ax1.set_ylabel('Acceptance Rate')
ax1.set_title('Acceptance Rate vs Draft Length (K)')
ax1.set_ylim(0, 1.0)
ax1.grid(True, alpha=0.3, axis='y')

ax2.bar(range(len(k_values)), k_speedups, tick_label=[str(k) for k in k_values], color='coral')
ax2.set_xlabel('K (draft tokens per iteration)')
ax2.set_ylabel('Speedup over standard decoding')
ax2.set_title('Speedup vs Draft Length (K)')
ax2.axhline(y=1.0, color='gray', linestyle='--', label='No speedup')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 2. Continuous Batching — Dynamic Batching for Throughput

In a serving scenario, multiple requests arrive at different times and require different numbers of output tokens. **Static batching** groups requests together and waits for the longest one to finish before returning any results. This wastes GPU cycles because shorter requests sit idle after completing.

**Continuous batching** (also called "iteration-level batching" or "inflight batching") solves this by managing the batch at the level of individual generation steps rather than complete requests. When a request finishes (hits EOS or max tokens), its slot is immediately freed and a new request from the queue can join the batch on the very next iteration.

The impact on throughput is dramatic. Consider a batch of 8 requests where the shortest needs 10 tokens and the longest needs 200. With static batching, all 8 slots are occupied for 200 steps, even though 7 of them are producing padding for most of that time. With continuous batching, those slots are recycled to serve new requests as soon as each finishes.

vLLM (Kwon et al., 2023) pioneered this approach in the open-source ecosystem and showed throughput improvements of 2-4x over static batching in realistic serving scenarios. Today, all major inference servers (vLLM, TGI, TensorRT-LLM) implement continuous batching.

In [ ]:
import random
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class Request:
    """A single inference request."""
    id: int
    prompt_tokens: int
    target_tokens: int  # how many tokens this request needs to generate
    generated: int = 0
    arrival_time: float = 0.0
    start_time: Optional[float] = None
    end_time: Optional[float] = None
    
    @property
    def done(self):
        return self.generated >= self.target_tokens
    
    @property
    def latency(self):
        if self.end_time and self.start_time:
            return self.end_time - self.start_time
        return None


def simulate_static_batching(requests: List[Request], max_batch_size: int,
                             time_per_step: float = 0.01):
    """
    Simulate static batching: fill a batch, run until ALL done, repeat.
    """
    completed = []
    queue = list(requests)
    current_time = 0.0
    total_steps = 0
    total_idle_slots = 0
    
    while queue or completed:
        if not queue:
            break
        
        # Fill batch
        batch = []
        while queue and len(batch) < max_batch_size:
            req = queue.pop(0)
            req.start_time = current_time
            batch.append(req)
        
        # Find max tokens needed in this batch
        max_tokens = max(r.target_tokens for r in batch)
        
        # Run batch for max_tokens steps
        for step in range(max_tokens):
            total_steps += 1
            current_time += time_per_step
            
            active_in_step = 0
            for req in batch:
                if not req.done:
                    req.generated += 1
                    active_in_step += 1
                    if req.done:
                        req.end_time = current_time
            
            total_idle_slots += (len(batch) - active_in_step)
        
        completed.extend(batch)
    
    return completed, {
        "total_steps": total_steps,
        "total_time": current_time,
        "total_idle_slots": total_idle_slots,
        "gpu_utilization": 1.0 - (total_idle_slots / (total_steps * max_batch_size))
    }


def simulate_continuous_batching(requests: List[Request], max_batch_size: int,
                                  time_per_step: float = 0.01):
    """
    Simulate continuous batching: replace finished requests immediately.
    """
    completed = []
    queue = list(requests)
    batch = []
    current_time = 0.0
    total_steps = 0
    total_idle_slots = 0
    
    while queue or batch:
        # Fill any empty slots from queue
        while queue and len(batch) < max_batch_size:
            req = queue.pop(0)
            req.start_time = current_time
            batch.append(req)
        
        if not batch:
            break
        
        # Run one step
        total_steps += 1
        current_time += time_per_step
        total_idle_slots += (max_batch_size - len(batch))
        
        # Process each request in the batch
        finished_this_step = []
        for req in batch:
            req.generated += 1
            if req.done:
                req.end_time = current_time
                finished_this_step.append(req)
        
        # Remove finished requests (slots become available next step)
        for req in finished_this_step:
            batch.remove(req)
            completed.append(req)
    
    return completed, {
        "total_steps": total_steps,
        "total_time": current_time,
        "total_idle_slots": total_idle_slots,
        "gpu_utilization": 1.0 - (total_idle_slots / (total_steps * max_batch_size)) if total_steps > 0 else 0
    }


# Generate realistic workload
random.seed(42)
n_requests = 50
requests_static = [
    Request(
        id=i,
        prompt_tokens=random.randint(10, 100),
        target_tokens=random.randint(10, 200),  # Highly variable output lengths
    )
    for i in range(n_requests)
]
# Deep copy for continuous batching
requests_continuous = [
    Request(id=r.id, prompt_tokens=r.prompt_tokens, target_tokens=r.target_tokens)
    for r in requests_static
]

max_batch = 8

completed_static, stats_static = simulate_static_batching(requests_static, max_batch)
completed_cont, stats_cont = simulate_continuous_batching(requests_continuous, max_batch)

print("=" * 60)
print(f"{'Metric':<30} {'Static':>12} {'Continuous':>12}")
print("=" * 60)
print(f"{'Total time':<30} {stats_static['total_time']:>12.3f} {stats_cont['total_time']:>12.3f}")
print(f"{'Total steps':<30} {stats_static['total_steps']:>12} {stats_cont['total_steps']:>12}")
print(f"{'GPU utilization':<30} {stats_static['gpu_utilization']:>11.1%} {stats_cont['gpu_utilization']:>11.1%}")
print(f"{'Avg latency':<30} {np.mean([r.latency for r in completed_static]):>12.3f} {np.mean([r.latency for r in completed_cont]):>12.3f}")
print(f"{'Throughput (req/s)':<30} {n_requests/stats_static['total_time']:>12.1f} {n_requests/stats_cont['total_time']:>12.1f}")

In [ ]:
# Visualize the difference
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Latency distribution
ax = axes[0]
static_latencies = [r.latency for r in completed_static]
cont_latencies = [r.latency for r in completed_cont]
ax.hist(static_latencies, bins=20, alpha=0.6, label='Static', color='red')
ax.hist(cont_latencies, bins=20, alpha=0.6, label='Continuous', color='green')
ax.set_xlabel('Latency (s)')
ax.set_ylabel('Count')
ax.set_title('Latency Distribution')
ax.legend()

# Throughput comparison
ax = axes[1]
methods = ['Static', 'Continuous']
throughputs = [
    n_requests / stats_static['total_time'],
    n_requests / stats_cont['total_time']
]
bars = ax.bar(methods, throughputs, color=['red', 'green'], alpha=0.7)
ax.set_ylabel('Requests / second')
ax.set_title('Throughput Comparison')
for bar, val in zip(bars, throughputs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', fontsize=12)

# GPU utilization
ax = axes[2]
utils = [stats_static['gpu_utilization'], stats_cont['gpu_utilization']]
bars = ax.bar(methods, utils, color=['red', 'green'], alpha=0.7)
ax.set_ylabel('GPU Utilization')
ax.set_title('GPU Utilization')
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, utils):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.1%}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

---
## 3. Paged Attention — Virtual Memory for KV Cache

As we saw in the foundations notebook, the KV cache can consume enormous amounts of GPU memory. The naive approach allocates a contiguous block of memory for each request's KV cache, sized for the maximum possible sequence length. This leads to two problems:

1. **Internal fragmentation**: If a request only generates 50 tokens but we allocated space for 2048, 97.5% of the memory is wasted.
2. **External fragmentation**: As requests of different lengths start and finish, the free memory becomes fragmented into small non-contiguous blocks that can't be used for new requests.

**PagedAttention** (Kwon et al., 2023, introduced in vLLM) borrows the concept of virtual memory from operating systems. Instead of allocating one contiguous block per request, it divides the KV cache into fixed-size **pages** (also called "blocks"). A request's KV cache is stored across potentially non-contiguous pages, with a page table mapping logical positions to physical memory locations.

This approach virtually eliminates memory waste:
- No internal fragmentation: pages are allocated on demand as the sequence grows
- No external fragmentation: any free page can be used by any request
- Memory sharing: for beam search or parallel sampling, multiple sequences can share the same KV cache pages for their common prefix (copy-on-write)

vLLM reports near-zero waste (< 4%) compared to 60-80% waste with naive allocation.

In [ ]:
class PagedKVCache:
    """
    Conceptual implementation of a paged KV cache.
    
    This demonstrates the core data structure, not the actual GPU kernel.
    In practice, the attention kernel must be aware of paging to
    gather K/V from non-contiguous pages.
    """
    
    def __init__(self, num_pages, page_size, kv_dim, n_layers):
        """
        Args:
            num_pages: total pages in the pool
            page_size: tokens per page (e.g., 16)
            kv_dim: dimension of K/V vectors
            n_layers: number of transformer layers
        """
        self.num_pages = num_pages
        self.page_size = page_size
        self.kv_dim = kv_dim
        self.n_layers = n_layers
        
        # Physical page pool: (num_pages, page_size, kv_dim) per layer, for K and V
        self.k_pages = torch.zeros(n_layers, num_pages, page_size, kv_dim)
        self.v_pages = torch.zeros(n_layers, num_pages, page_size, kv_dim)
        
        # Track which pages are free
        self.free_pages = list(range(num_pages))
        
        # Page tables: request_id -> list of physical page indices
        self.page_tables = {}
        
        # Track how many tokens are in each request's last page
        self.tokens_in_last_page = {}
    
    def allocate(self, request_id):
        """Initialize a page table for a new request."""
        self.page_tables[request_id] = []
        self.tokens_in_last_page[request_id] = 0
    
    def append_token(self, request_id, layer, k_vector, v_vector):
        """
        Append one token's K/V to the cache.
        Allocates a new page if the current one is full.
        """
        pages = self.page_tables[request_id]
        tokens_in_last = self.tokens_in_last_page[request_id]
        
        # Need a new page?
        if not pages or tokens_in_last >= self.page_size:
            if not self.free_pages:
                raise RuntimeError("Out of pages! Need to evict or wait.")
            new_page = self.free_pages.pop(0)
            pages.append(new_page)
            self.tokens_in_last_page[request_id] = 0
            tokens_in_last = 0
        
        # Write K/V into the current page
        physical_page = pages[-1]
        slot = tokens_in_last
        self.k_pages[layer, physical_page, slot] = k_vector
        self.v_pages[layer, physical_page, slot] = v_vector
        self.tokens_in_last_page[request_id] = slot + 1
    
    def get_kv(self, request_id, layer):
        """Retrieve all K/V for a request by gathering from pages."""
        pages = self.page_tables[request_id]
        if not pages:
            return None, None
        
        k_list, v_list = [], []
        for i, page_idx in enumerate(pages):
            if i < len(pages) - 1:
                # Full page
                k_list.append(self.k_pages[layer, page_idx])
                v_list.append(self.v_pages[layer, page_idx])
            else:
                # Last page — might be partial
                n = self.tokens_in_last_page[request_id]
                k_list.append(self.k_pages[layer, page_idx, :n])
                v_list.append(self.v_pages[layer, page_idx, :n])
        
        return torch.cat(k_list, dim=0), torch.cat(v_list, dim=0)
    
    def free(self, request_id):
        """Release all pages for a completed request."""
        pages = self.page_tables.pop(request_id, [])
        self.free_pages.extend(pages)
        self.tokens_in_last_page.pop(request_id, None)
    
    @property
    def utilization(self):
        used = self.num_pages - len(self.free_pages)
        return used / self.num_pages


# Demo
cache = PagedKVCache(num_pages=100, page_size=16, kv_dim=64, n_layers=2)
print(f"Total pages: {cache.num_pages}, Page size: {cache.page_size} tokens")
print(f"Total capacity: {cache.num_pages * cache.page_size} tokens")
print(f"Initial utilization: {cache.utilization:.1%}")

# Simulate 3 requests with different lengths
cache.allocate("req_1")
cache.allocate("req_2")
cache.allocate("req_3")

# Request 1: 10 tokens, Request 2: 35 tokens, Request 3: 5 tokens
for i in range(35):
    k_vec = torch.randn(64)
    v_vec = torch.randn(64)
    for layer in range(2):
        if i < 10:
            cache.append_token("req_1", layer, k_vec, v_vec)
        cache.append_token("req_2", layer, k_vec, v_vec)
        if i < 5:
            cache.append_token("req_3", layer, k_vec, v_vec)

print(f"\nAfter filling:")
print(f"  req_1: {len(cache.page_tables['req_1'])} pages (10 tokens)")
print(f"  req_2: {len(cache.page_tables['req_2'])} pages (35 tokens)")
print(f"  req_3: {len(cache.page_tables['req_3'])} pages (5 tokens)")
print(f"  Utilization: {cache.utilization:.1%}")

# Free request 1
cache.free("req_1")
print(f"\nAfter freeing req_1: {cache.utilization:.1%} utilization")
print(f"Free pages: {len(cache.free_pages)}")

In [ ]:
# Compare memory waste: contiguous vs paged allocation
def simulate_memory_waste(n_requests, max_seq_len, actual_lengths, page_size=16):
    """Compare memory waste between contiguous and paged allocation."""
    # Contiguous: allocate max_seq_len for each request
    contiguous_allocated = n_requests * max_seq_len
    contiguous_used = sum(actual_lengths)
    contiguous_waste = 1.0 - (contiguous_used / contiguous_allocated)
    
    # Paged: allocate ceil(actual_length / page_size) pages per request
    import math
    paged_pages = sum(math.ceil(l / page_size) for l in actual_lengths)
    paged_allocated = paged_pages * page_size
    paged_waste = 1.0 - (contiguous_used / paged_allocated)
    
    return contiguous_waste, paged_waste


# Simulate with different length distributions
random.seed(42)
n_req = 100
max_len = 2048

# Distribution 1: uniform
uniform_lengths = [random.randint(10, max_len) for _ in range(n_req)]
# Distribution 2: skewed short
short_lengths = [min(random.expovariate(1/100), max_len) for _ in range(n_req)]
short_lengths = [max(10, int(l)) for l in short_lengths]
# Distribution 3: bimodal
bimodal_lengths = [random.choice([random.randint(10, 100), random.randint(1000, 2000)]) for _ in range(n_req)]

fig, ax = plt.subplots(figsize=(10, 6))

distributions = {
    "Uniform (10-2048)": uniform_lengths,
    "Skewed short": short_lengths,
    "Bimodal": bimodal_lengths,
}

x = np.arange(len(distributions))
width = 0.35
cont_wastes = []
paged_wastes = []

for name, lengths in distributions.items():
    cw, pw = simulate_memory_waste(n_req, max_len, lengths)
    cont_wastes.append(cw)
    paged_wastes.append(pw)

bars1 = ax.bar(x - width/2, cont_wastes, width, label='Contiguous', color='red', alpha=0.7)
bars2 = ax.bar(x + width/2, paged_wastes, width, label='Paged (16 tokens/page)', color='green', alpha=0.7)

ax.set_ylabel('Memory Waste')
ax.set_title('Memory Waste: Contiguous vs Paged KV Cache')
ax.set_xticks(x)
ax.set_xticklabels(distributions.keys())
ax.legend()
ax.set_ylim(0, 1.0)

for bar, val in zip(bars1, cont_wastes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.0%}', ha='center')
for bar, val in zip(bars2, paged_wastes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.0%}', ha='center')

plt.tight_layout()
plt.show()

---
## 4. Beam Search Variants — Diverse and Group Beam Search

Standard beam search has a well-known problem: the beams tend to converge to very similar sequences. If the top beam starts with "The cat sat on", most other beams will also start similarly because they share the same high-probability prefix. This means beam search explores only a narrow region of the output space.

**Diverse beam search** (Vijayakumar et al., 2016) addresses this by adding a diversity penalty. Beams are divided into groups, and within each step, tokens that were already selected by earlier groups receive a penalty. This forces later groups to explore different regions of the output space.

The diversity penalty is controlled by a parameter (often called `diversity_penalty` or `lambda`). At 0, it degenerates to standard beam search. At high values, the beams become maximally diverse but may sacrifice quality.

In [ ]:
def diverse_beam_search(model, tokenizer, prompt, max_new_tokens=30,
                        num_beams=6, num_groups=3, diversity_penalty=1.0):
    """
    Diverse beam search: divide beams into groups and penalize overlap.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    beams_per_group = num_beams // num_groups
    
    # Initialize: each group has beams_per_group beams
    # Each beam: (score, sequence)
    groups = [
        [(0.0, input_ids[0].clone())]
        for _ in range(num_groups)
    ]
    
    for step in range(max_new_tokens):
        # Track which tokens were selected by earlier groups this step
        selected_tokens_by_previous_groups = []
        
        new_groups = []
        for g_idx, group_beams in enumerate(groups):
            all_candidates = []
            
            for score, seq in group_beams:
                if seq[-1].item() == tokenizer.eos_token_id:
                    all_candidates.append((score, seq))
                    continue
                
                with torch.no_grad():
                    logits = model(seq.unsqueeze(0).to(device)).logits[0, -1, :]
                log_probs = F.log_softmax(logits, dim=-1)
                
                # Apply diversity penalty for tokens selected by previous groups
                if selected_tokens_by_previous_groups:
                    for prev_token in selected_tokens_by_previous_groups:
                        log_probs[prev_token] -= diversity_penalty
                
                topk_scores, topk_ids = torch.topk(log_probs, beams_per_group * 2)
                
                for i in range(beams_per_group * 2):
                    new_score = score + topk_scores[i].item()
                    new_seq = torch.cat([seq, topk_ids[i:i+1]])
                    all_candidates.append((new_score, new_seq))
            
            # Keep top beams for this group
            all_candidates.sort(key=lambda x: x[0], reverse=True)
            new_group = all_candidates[:beams_per_group]
            new_groups.append(new_group)
            
            # Record which tokens this group selected
            for _, seq in new_group:
                selected_tokens_by_previous_groups.append(seq[-1].item())
        
        groups = new_groups
    
    # Collect all final beams
    all_beams = []
    for g_idx, group in enumerate(groups):
        for score, seq in group:
            text = tokenizer.decode(seq, skip_special_tokens=True)
            all_beams.append((score / len(seq), text, g_idx))
    
    all_beams.sort(key=lambda x: x[0], reverse=True)
    return all_beams


prompt = "The future of artificial intelligence is"

# Standard beam search (diversity_penalty=0)
print("=== Standard Beam Search (no diversity) ===")
standard_beams = diverse_beam_search(
    target_model, tokenizer, prompt,
    max_new_tokens=20, num_beams=6, num_groups=3, diversity_penalty=0.0
)
for score, text, group in standard_beams[:6]:
    print(f"  [Group {group}] (score: {score:.3f}) {text}")

print("\n=== Diverse Beam Search (penalty=1.0) ===")
diverse_beams = diverse_beam_search(
    target_model, tokenizer, prompt,
    max_new_tokens=20, num_beams=6, num_groups=3, diversity_penalty=1.0
)
for score, text, group in diverse_beams[:6]:
    print(f"  [Group {group}] (score: {score:.3f}) {text}")

---
## 5. Repetition Penalties — Frequency and Presence Penalties

Autoregressive language models have a strong tendency toward repetition. Once a phrase or pattern appears in the generated text, the model often assigns it even higher probability, creating a feedback loop. This is especially problematic with greedy decoding but occurs with sampling too.

Two types of penalties address this:

**Frequency penalty**: Penalizes tokens proportionally to how many times they've appeared. A token that appeared 3 times gets 3x the penalty of one that appeared once. This discourages common words from appearing too often but still allows them.

**Presence penalty**: Penalizes any token that has appeared at all, regardless of count. This encourages the model to talk about new topics rather than revisiting old ones. It's a binary signal: you either appeared or you didn't.

OpenAI's API exposes both as separate parameters. The mathematical formulation modifies logits as:
```
logit_modified = logit - frequency_penalty * count(token) - presence_penalty * (1 if count(token) > 0 else 0)
```

In [ ]:
from collections import Counter


def generate_with_penalties(
    model, tokenizer, prompt, max_new_tokens=80,
    temperature=0.8, frequency_penalty=0.0, presence_penalty=0.0
):
    """
    Generate with frequency and presence penalties on repeated tokens.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    token_counts = Counter()
    
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated).logits[:, -1, :]  # (1, vocab_size)
        
        # Apply penalties
        if frequency_penalty > 0 or presence_penalty > 0:
            for token_id, count in token_counts.items():
                logits[0, token_id] -= frequency_penalty * count
                logits[0, token_id] -= presence_penalty * (1 if count > 0 else 0)
        
        # Sample
        probs = F.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        token_id = next_token[0, 0].item()
        
        token_counts[token_id] += 1
        generated = torch.cat([generated, next_token], dim=-1)
        
        if token_id == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(generated[0], skip_special_tokens=True)


prompt = "The most important thing about machine learning is"

print("=== No penalties ===")
print(generate_with_penalties(target_model, tokenizer, prompt, frequency_penalty=0, presence_penalty=0))

print("\n=== Frequency penalty = 1.0 ===")
print(generate_with_penalties(target_model, tokenizer, prompt, frequency_penalty=1.0, presence_penalty=0))

print("\n=== Presence penalty = 1.0 ===")
print(generate_with_penalties(target_model, tokenizer, prompt, frequency_penalty=0, presence_penalty=1.0))

print("\n=== Both penalties = 0.5 ===")
print(generate_with_penalties(target_model, tokenizer, prompt, frequency_penalty=0.5, presence_penalty=0.5))

In [ ]:
# Measure vocabulary diversity under different penalties
def measure_diversity(text, tokenizer):
    tokens = tokenizer.encode(text)
    unique = len(set(tokens))
    total = len(tokens)
    return unique / total if total > 0 else 0


penalties = [0, 0.2, 0.5, 1.0, 1.5, 2.0]
freq_diversities = []
pres_diversities = []

for p in penalties:
    # Average over a few runs
    fd, pd = [], []
    for _ in range(3):
        text_f = generate_with_penalties(target_model, tokenizer, prompt, 60, frequency_penalty=p)
        text_p = generate_with_penalties(target_model, tokenizer, prompt, 60, presence_penalty=p)
        fd.append(measure_diversity(text_f, tokenizer))
        pd.append(measure_diversity(text_p, tokenizer))
    freq_diversities.append(np.mean(fd))
    pres_diversities.append(np.mean(pd))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(penalties, freq_diversities, 'b-o', label='Frequency penalty', linewidth=2)
ax.plot(penalties, pres_diversities, 'r-s', label='Presence penalty', linewidth=2)
ax.set_xlabel('Penalty value')
ax.set_ylabel('Vocabulary diversity (unique/total tokens)')
ax.set_title('Repetition Penalty Effect on Vocabulary Diversity')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Logit Processors — Custom Logit Manipulation Pipeline

All the techniques we've covered — temperature, top-k, top-p, repetition penalties, constrained decoding — share a common pattern: they modify the logits before token selection. HuggingFace formalizes this with the **LogitsProcessor** abstraction.

A `LogitsProcessor` is any callable that takes `(input_ids, scores)` and returns modified `scores`. Multiple processors can be chained into a `LogitsProcessorList`, which applies them sequentially. This composability is powerful: you can combine temperature scaling, repetition penalty, banned tokens, and custom constraints in any order.

This pattern is also how inference frameworks implement "logit bias" — the ability to boost or suppress specific tokens via the API. When you set `logit_bias={"42": 5.0}` in an OpenAI API call, you're adding a logit processor that adds 5.0 to token 42's logit.

In [ ]:
from transformers import LogitsProcessor, LogitsProcessorList


class BannedTokensProcessor(LogitsProcessor):
    """Prevent specific tokens from being generated."""
    
    def __init__(self, banned_token_ids):
        self.banned_token_ids = banned_token_ids
    
    def __call__(self, input_ids, scores):
        for token_id in self.banned_token_ids:
            scores[:, token_id] = float('-inf')
        return scores


class ForcedTokenProcessor(LogitsProcessor):
    """Force a specific token at a specific step."""
    
    def __init__(self, step_token_map, prompt_length):
        """
        step_token_map: dict of {generation_step: token_id}
        """
        self.step_token_map = step_token_map
        self.prompt_length = prompt_length
    
    def __call__(self, input_ids, scores):
        current_step = input_ids.shape[1] - self.prompt_length
        if current_step in self.step_token_map:
            forced_id = self.step_token_map[current_step]
            scores[:, :] = float('-inf')
            scores[:, forced_id] = 0
        return scores


class RepetitionPenaltyProcessor(LogitsProcessor):
    """Apply multiplicative repetition penalty."""
    
    def __init__(self, penalty=1.2):
        self.penalty = penalty
    
    def __call__(self, input_ids, scores):
        for batch_idx in range(input_ids.shape[0]):
            for token_id in set(input_ids[batch_idx].tolist()):
                if scores[batch_idx, token_id] > 0:
                    scores[batch_idx, token_id] /= self.penalty
                else:
                    scores[batch_idx, token_id] *= self.penalty
        return scores


class TemperatureProcessor(LogitsProcessor):
    """Apply temperature scaling."""
    
    def __init__(self, temperature=1.0):
        self.temperature = temperature
    
    def __call__(self, input_ids, scores):
        return scores / self.temperature


# Build a processor pipeline
prompt = "Python is a great programming language because"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# Ban certain words by finding their token IDs
banned_words = [" Java", " C++", " JavaScript"]
banned_ids = [tokenizer.encode(w)[0] for w in banned_words]

processors = LogitsProcessorList([
    BannedTokensProcessor(banned_ids),
    RepetitionPenaltyProcessor(penalty=1.3),
    TemperatureProcessor(temperature=0.7),
])

# Generate using the pipeline
output = target_model.generate(
    input_ids,
    max_new_tokens=60,
    logits_processor=processors,
    do_sample=True,
)
print("With custom logit processor pipeline:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# Demonstrate composability: stack multiple processors
class TopicBoostProcessor(LogitsProcessor):
    """Boost logits for tokens related to a desired topic."""
    
    def __init__(self, tokenizer, topic_words, boost=3.0):
        self.boost = boost
        self.topic_token_ids = set()
        for word in topic_words:
            ids = tokenizer.encode(word)
            self.topic_token_ids.update(ids)
    
    def __call__(self, input_ids, scores):
        for token_id in self.topic_token_ids:
            scores[:, token_id] += self.boost
        return scores


# Steer generation toward space/science topics
space_words = [" space", " stars", " galaxy", " universe", " planets", " cosmic",
               " orbit", " rocket", " NASA", " telescope", " solar"]

processors_steered = LogitsProcessorList([
    TopicBoostProcessor(tokenizer, space_words, boost=5.0),
    RepetitionPenaltyProcessor(penalty=1.2),
    TemperatureProcessor(temperature=0.8),
])

prompt_neutral = "The most exciting discovery of the century was"
input_ids = tokenizer.encode(prompt_neutral, return_tensors="pt").to(device)

# Without topic steering
output_plain = target_model.generate(
    input_ids, max_new_tokens=50, do_sample=True,
    logits_processor=LogitsProcessorList([TemperatureProcessor(0.8)])
)
print("Without topic steering:")
print(tokenizer.decode(output_plain[0], skip_special_tokens=True))

# With space topic steering
output_steered = target_model.generate(
    input_ids, max_new_tokens=50, do_sample=True,
    logits_processor=processors_steered
)
print("\nWith space topic steering:")
print(tokenizer.decode(output_steered[0], skip_special_tokens=True))

---
## Summary

In this notebook we covered advanced inference techniques that form the backbone of modern serving infrastructure:

1. **Speculative decoding**: Use a small draft model to propose multiple tokens, verify them in parallel with the target model. Achieves 2-3x speedup with zero quality loss — the output distribution is mathematically identical to standard decoding.

2. **Continuous batching**: Replace finished requests immediately instead of waiting for the entire batch. Dramatically improves GPU utilization and throughput in serving scenarios.

3. **Paged attention**: Manage KV cache memory like an OS manages virtual memory — non-contiguous pages, on-demand allocation, copy-on-write sharing. Reduces memory waste from 60-80% to under 4%.

4. **Diverse beam search**: Force beam groups to explore different regions of the output space by penalizing tokens already selected by other groups.

5. **Repetition penalties**: Frequency penalty (proportional to count) and presence penalty (binary) keep the model from getting stuck in loops.

6. **Logit processors**: A composable pipeline for logit manipulation. All decoding modifications (temperature, penalties, constraints, biases) can be expressed as logit processors.

**Next: [expert.ipynb](./expert.ipynb)** — Quantization (GPTQ, AWQ, GGUF), kernel optimization, and inference benchmarking.